## Rule Profile & Topic Pool 생성 파이프라인

`1_topic_tagging_v5`에서 **Rule Profile** 및 **Topic Pool** 생성 로직만 추출한 노트북.

### 실행 순서
1. Config & Imports
2. Helpers (LLM 호출, JSON 파싱, 저장)
3. Category Feature Patterns
4. 대상 그룹 추출 & 샘플링 함수
5. Checkpoint/Resume 유틸
6. **Rule Profile 생성** — 그룹별 분류 규칙 설계
7. **Topic Pool 생성** — 규칙 기반 주제 taxonomy 설계

In [0]:
# 1) Config & Imports

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"
ENDPOINT = "databricks-gpt-5-4"

PROMPT_VERSION = "category_topic_dynamic_rules_allgroups_v1"
SAVE_DB = "sandbox.z_jungryo_lee"

RULE_PROFILE_TABLE = f"{SAVE_DB}.category_topic_rule_profile_{PROMPT_VERSION}"
TOPIC_POOL_TABLE = f"{SAVE_DB}.category_topic_pool_{PROMPT_VERSION}"

PROGRESS_TABLE = f"{SAVE_DB}.category_topic_progress_{PROMPT_VERSION}"
FAILED_GROUP_TABLE = f"{SAVE_DB}.category_topic_failed_group_{PROMPT_VERSION}"

MIN_GROUP_SIZE = 100
MAX_SAMPLE_ROWS = 1000
MAX_RULE_SAMPLE_ROWS = 800
MAX_FINAL_TOPICS = 17
MIN_FINAL_TOPICS = 7

MAX_TOKENS = 2200
MAX_RETRIES = 3
BACKOFF_BASE = 1.8
RATE_LIMIT_SECONDS = 0.4

RESUME_FROM_CHECKPOINT = True
LIMIT_GROUP_COUNT = None  # None = 전체, 숫자 = 테스트

print(f"[CONFIG] endpoint={ENDPOINT}, prompt_version={PROMPT_VERSION}")
print(f"[CONFIG] rule_sample={MAX_RULE_SAMPLE_ROWS}, topic_sample={MAX_SAMPLE_ROWS}, topics={MIN_FINAL_TOPICS}~{MAX_FINAL_TOPICS}")

In [0]:
# 2) Helpers

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)
    raise ValueError(f"Invalid JSON: {text[:1000]}")


def sc_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "긍정"
    if sc_measurement == -1:
        return "부정"
    return "기타"


def overall_label(sc_measurement: int) -> str:
    if sc_measurement == 1:
        return "전반적 긍정"
    if sc_measurement == -1:
        return "전반적 부정"
    return "전반적 평가"


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client

    client = get_deploy_client("databricks")
    payload = {"messages": messages, "temperature": 0.0, "max_tokens": max_tokens}

    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            time.sleep(BACKOFF_BASE ** attempt)
    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


def save_table(df, table_name: str) -> None:
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)
    print(f"[SAVE] overwrite -> {table_name}")


def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)

In [0]:
# 3) Category Feature Patterns (축약 - 주요 카테고리만)

COMMON_FEATURE_PATTERNS = [
    "quality", "picture", "image", "visual", "sound", "audio", "video", "screen", "display", "tv",
    "setup", "install", "cable", "port", "hdmi", "bluetooth", "wifi", "wireless", "connect",
    "os", "software", "menu", "ui", "app", "channel", "content", "voice", "game", "gaming",
    "iot", "mobile", "brand", "service", "price", "design", "color", "heat", "durability",
    "화질", "음질", "사운드", "화면", "설치", "세팅", "블루투스", "와이파이", "연결",
    "메뉴", "앱", "채널", "콘텐츠", "음성", "게임", "iot", "모바일", "브랜드",
    "가격", "가성비", "디자인", "색상", "소재", "마감", "발열", "내구성", "해상도"
]

CATEGORY_FEATURE_PATTERNS = {
    ("07. 스마트 사용성", "07-01. 채널/컨텐츠 APP"): [
        "app", "apps", "channel", "content", "streaming", "ott",
        "앱", "채널", "콘텐츠", "스트리밍", "ott"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(1)TV 전반"): [
        "fast", "slow", "lag", "speed", "tv response",
        "속도", "빠름", "느림", "반응", "렉"
    ],
    ("07. 스마트 사용성", "07-03. 메뉴 구성/UI"): [
        "menu", "ui", "interface", "navigation",
        "메뉴", "ui", "인터페이스", "탐색"
    ],
    ("07. 스마트 사용성", "07-04. SW/OS"): [
        "os", "software", "update", "bug",
        "os", "소프트웨어", "업데이트", "버그"
    ],
    ("07. 스마트 사용성", "07-06. 리모컨 사용성"): [
        "remote", "control", "button", "pointer",
        "리모컨", "조작", "버튼", "포인터"
    ],
    ("07. 스마트 사용성", "07-08. 음성 인식/조작"): [
        "voice", "speech", "assistant",
        "음성", "음성인식", "음성조작"
    ],
    ("07. 스마트 사용성", "07-09. 게임 기능"): [
        "game", "gaming", "latency",
        "게임", "게이밍", "지연"
    ],
    ("13. 전반적 평가", "13-01. 전반적 평가"): [
        "overall", "generally", "satisfied", "happy", "purchase",
        "전반적", "전체적", "만족", "구매", "제품"
    ],
}
# NOTE: 전체 패턴은 원본 노트북(1_topic_tagging_v5 Cell 3) 참조

def get_feature_patterns(cate_1_depth: str, cate_2_depth: str) -> List[str]:
    return COMMON_FEATURE_PATTERNS + CATEGORY_FEATURE_PATTERNS.get((cate_1_depth, cate_2_depth), [])

In [0]:
# 4) 대상 그룹 추출 & 샘플링 함수

group_query = f"""
with base as (
    select distinct cate_1_depth, cate_2_depth, sc_measurement, memo
    from {SOURCE_TABLE}
    where cate_1_depth not like '***%'
      and sc_measurement in (1, -1)
      and memo is not null
      and length(trim(memo)) > 0
),
grouped as (
    select cate_1_depth, cate_2_depth, sc_measurement, count(*) as group_total_cnt
    from base
    group by 1, 2, 3
)
select * from grouped
where group_total_cnt >= {MIN_GROUP_SIZE}
order by cate_1_depth, cate_2_depth, sc_measurement
"""

group_df = spark.sql(group_query)
if LIMIT_GROUP_COUNT is not None:
    group_df = group_df.limit(LIMIT_GROUP_COUNT)

group_rows = [r.asDict(recursive=True) for r in group_df.toLocalIterator()]
print(f"[LOAD] total groups = {len(group_rows)}")
display(group_df)


def load_group_sample_df(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, max_rows: int):
    """고정 seed 기반 재현 가능한 샘플링"""
    query = f"""
    with base as (
        select distinct memo
        from {SOURCE_TABLE}
        where cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
          and memo is not null
          and length(trim(memo)) > 0
    ),
    sampled as (
        select memo,
               row_number() over (
                   order by md5(concat(
                       coalesce(memo, ''), '||',
                       '{cate_1_depth}', '||',
                       '{cate_2_depth}', '||',
                       cast({sc_measurement} as string),
                       '||seed_20260420'
                   ))
               ) as rn
        from base
    )
    select memo from sampled where rn <= {max_rows} order by rn
    """
    return spark.sql(query).withColumn("_row_id", F.monotonically_increasing_id()).cache()

In [0]:
# 5) Checkpoint / Resume Helpers

PROGRESS_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

FAILED_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])


def ensure_checkpoint_tables() -> None:
    for tbl, schema_str in [
        (PROGRESS_TABLE, "cate_1_depth string, cate_2_depth string, sc_measurement int, stage string, status string, message string, event_ts timestamp"),
        (FAILED_GROUP_TABLE, "cate_1_depth string, cate_2_depth string, sc_measurement int, stage string, error_message string, event_ts timestamp"),
    ]:
        if not spark.catalog.tableExists(tbl):
            spark.createDataFrame([], schema_str).write.mode("overwrite").format("delta").saveAsTable(tbl)


def log_progress(cate_1_depth, cate_2_depth, sc_measurement, stage, status, message=""):
    row = [(clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
            clean_text(stage), clean_text(status), clean_text(message), pd.Timestamp.now().to_pydatetime())]
    append_table(spark.createDataFrame(row, schema=PROGRESS_STRUCT), PROGRESS_TABLE)


def log_failure(cate_1_depth, cate_2_depth, sc_measurement, stage, error_message):
    row = [(clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
            clean_text(stage), clean_text(error_message), pd.Timestamp.now().to_pydatetime())]
    append_table(spark.createDataFrame(row, schema=FAILED_STRUCT), FAILED_GROUP_TABLE)


def get_done_group_keys(stage: str):
    if (not RESUME_FROM_CHECKPOINT) or (not spark.catalog.tableExists(PROGRESS_TABLE)):
        return set()
    rows = (
        spark.table(PROGRESS_TABLE)
        .where(F.col("stage") == F.lit(stage))
        .where(F.col("status") == F.lit("done"))
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .distinct().collect()
    )
    return {(r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])) for r in rows}


def delete_group_rows(table_name, cate_1_depth, cate_2_depth, sc_measurement):
    if not spark.catalog.tableExists(table_name):
        return
    spark.sql(f"""
        delete from {table_name}
        where cate_1_depth = '{cate_1_depth}'
          and cate_2_depth = '{cate_2_depth}'
          and sc_measurement = {sc_measurement}
    """)


ensure_checkpoint_tables()
print(f"[CHECKPOINT READY] {PROGRESS_TABLE}")

In [0]:
# 6) Build Rule Profile — 그룹별 분류 규칙 생성

spark.sql(f"truncate table {FAILED_GROUP_TABLE}")

def build_rule_profile_messages(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, sample_memos: List[str]) -> List[Dict[str, str]]:
    label = sc_label(sc_measurement)
    overall = overall_label(sc_measurement)
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC rule designer for TV review topic classification.

Category:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- polarity = {label}

Design outputs:
- overall_topic_label
- clue_keywords
- generic_terms
- overall_usage_rule
- specific_reason_rule
- non_overall_examples

Rules:
- clue_keywords must be specialized for this category.
- generic_terms must be suitable for this polarity only.
- {overall} should be extremely narrow.
- {overall} must be limited to ultra-short sentiment-only memos where no usable reason can be inferred.
- If a memo contains any usable reason, feature, benefit, complaint, or context clue, it must not be treated as {overall}.
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}

Return only JSON:
{{
  "overall_topic_label": "",
  "clue_keywords": [""],
  "generic_terms": [""],
  "overall_usage_rule": "",
  "specific_reason_rule": "",
  "non_overall_examples": [""]
}}
"""
    user = "Review memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


done_rule_groups = get_done_group_keys("rule_profile")
total_groups = len(group_rows)
print(f"[RULE PROFILE] total={total_groups}, done={len(done_rule_groups)}, pending={total_groups - len(done_rule_groups)}")

for idx, g in enumerate(group_rows, start=1):
    key = (g["cate_1_depth"], g["cate_2_depth"], int(g["sc_measurement"]))

    if key in done_rule_groups:
        continue

    print(f"[RULE PROFILE] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        rule_sample_df = load_group_sample_df(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], MAX_RULE_SAMPLE_ROWS)
        rule_sample_memos = [r["memo"] for r in rule_sample_df.select("memo").toLocalIterator()]

        rule_result = call_llm(
            build_rule_profile_messages(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], rule_sample_memos)
        )

        rule_pdf = pd.DataFrame([{
            "cate_1_depth": g["cate_1_depth"],
            "cate_2_depth": g["cate_2_depth"],
            "sc_measurement": g["sc_measurement"],
            "group_total_cnt": g["group_total_cnt"],
            "overall_topic_label": clean_text(rule_result.get("overall_topic_label")) or overall_label(g["sc_measurement"]),
            "clue_keywords_json": json.dumps(rule_result.get("clue_keywords", [])[:80], ensure_ascii=False),
            "generic_terms_json": json.dumps(rule_result.get("generic_terms", [])[:30], ensure_ascii=False),
            "overall_usage_rule": clean_text(rule_result.get("overall_usage_rule")),
            "specific_reason_rule": clean_text(rule_result.get("specific_reason_rule")),
            "non_overall_examples_json": json.dumps(rule_result.get("non_overall_examples", [])[:12], ensure_ascii=False),
        }])

        delete_group_rows(RULE_PROFILE_TABLE, g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"])
        append_table(spark.createDataFrame(rule_pdf), RULE_PROFILE_TABLE)
        log_progress(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "rule_profile", "done", "saved")
        rule_sample_df.unpersist()

        elapsed = round(time.time() - group_start_ts, 2)
        print(f"[RULE PROFILE DONE] {idx}/{total_groups} ({round(idx/total_groups*100,1)}%) {elapsed}s")
        time.sleep(RATE_LIMIT_SECONDS)

    except Exception as exc:
        log_failure(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "rule_profile", repr(exc))
        print(f"[RULE PROFILE FAILED] {idx}/{total_groups} error={repr(exc)}")

print("\n[RULE PROFILE 완료]")
display(spark.table(RULE_PROFILE_TABLE))

In [0]:
# 7) Build Topic Pool — 규칙 기반 주제 Taxonomy 생성

rule_map = {
    (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])): r.asDict(recursive=True)
    for r in spark.table(RULE_PROFILE_TABLE).toLocalIterator()
}

def build_topic_pool_messages(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, sample_memos: List[str], rule_row: Dict[str, Any]) -> List[Dict[str, str]]:
    overall_topic_label = clean_text(rule_row["overall_topic_label"])
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC taxonomy designer for TV review topic classification.

Category:
- cate_1_depth = {cate_1_depth}
- cate_2_depth = {cate_2_depth}
- polarity = {sc_label(sc_measurement)}

Rules:
- Final topic count must be at least {MIN_FINAL_TOPICS} and at most {MAX_FINAL_TOPICS}.
- One mandatory topic must be '{overall_topic_label}'.
- {clean_text(rule_row["overall_usage_rule"])}
- {clean_text(rule_row["specific_reason_rule"])}
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:80], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}
- Topic labels must be Korean.
- Topic should be concise noun-like or service/function-like wording.
- Description should be short Korean.
- Avoid duplicates and near-synonyms.

Return only JSON:
{{
  "topics": [
    {{
      "topic": "",
      "description": "",
      "representative_memos": [""]
    }}
  ]
}}
"""
    user = "Sample memos:\n" + "\n".join([f"- {clean_text(m)}" for m in sample_memos])
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


done_topic_groups = get_done_group_keys("topic_pool")
total_groups = len(group_rows)
print(f"[TOPIC POOL] total={total_groups}, done={len(done_topic_groups)}, pending={total_groups - len(done_topic_groups)}")

for idx, g in enumerate(group_rows, start=1):
    key = (g["cate_1_depth"], g["cate_2_depth"], int(g["sc_measurement"]))

    if key in done_topic_groups:
        continue
    if key not in rule_map:
        print(f"[TOPIC POOL SKIP] missing rule profile - {key}")
        continue

    print(f"[TOPIC POOL] {idx}/{total_groups} - {key}")
    group_start_ts = time.time()

    try:
        sample_df = load_group_sample_df(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], MAX_SAMPLE_ROWS)
        sample_memos = [r["memo"] for r in sample_df.select("memo").toLocalIterator()]

        topic_pool_result = call_llm(
            build_topic_pool_messages(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], sample_memos, rule_map[key])
        )

        topic_rows = []
        for order_no, topic in enumerate(topic_pool_result.get("topics", [])[:MAX_FINAL_TOPICS], start=1):
            if not isinstance(topic, dict):
                continue
            topic_rows.append({
                "cate_1_depth": g["cate_1_depth"],
                "cate_2_depth": g["cate_2_depth"],
                "sc_measurement": g["sc_measurement"],
                "topic_order": order_no,
                "topic": clean_text(topic.get("topic")),
                "description": clean_text(topic.get("description")),
                "representative_memos_json": json.dumps(topic.get("representative_memos", [])[:5], ensure_ascii=False),
            })

        delete_group_rows(TOPIC_POOL_TABLE, g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"])
        append_table(spark.createDataFrame(pd.DataFrame(topic_rows)), TOPIC_POOL_TABLE)
        log_progress(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "topic_pool", "done", "saved")
        sample_df.unpersist()

        elapsed = round(time.time() - group_start_ts, 2)
        print(f"[TOPIC POOL DONE] {idx}/{total_groups} ({round(idx/total_groups*100,1)}%) {elapsed}s")
        time.sleep(RATE_LIMIT_SECONDS)

    except Exception as exc:
        log_failure(g["cate_1_depth"], g["cate_2_depth"], g["sc_measurement"], "topic_pool", repr(exc))
        print(f"[TOPIC POOL FAILED] {idx}/{total_groups} error={repr(exc)}")

print("\n[TOPIC POOL 완료]")
display(spark.table(TOPIC_POOL_TABLE))